# R5 — answer_policy_question + final gold_answers (incl. unanswerable U01/U02)

Wraps the proven R4 pipeline (hybrid retrieval → LLM rerank → grounded answer) into one reusable function `answer_policy_question(q)`, runs it over the 19 gold-set questions **plus two unanswerable questions U01/U02**, and writes the final `claims.genai.gold_answers` table (`cited_chunk_ids` as ARRAY<STRING>, one row per question, 1–5 bare chunk ids in rank order).

- U01 (pet-dog premium) and U02 (gym membership) are covered nowhere in the binder: their answers must contain the phrase **"not in policy"**
- Graded questions (G01–G19) must NOT use that phrase unless the binder truly cannot answer them — all 19 are answerable here (requirement: ≥ 16 answered)

In [0]:
# Paper 4 config cell - change endpoint names HERE and nowhere else (same constants as R4)
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

INDEX_NAME = f"{CATALOG}.genai.policy_chunks_idx"
SOURCE_TABLE = f"{CATALOG}.genai.policy_chunks"
GOLD_ANSWERS = f"{CATALOG}.genai.gold_answers"
CLAIMS_TABLE = f"{CATALOG}.gold.claims_enriched"   # structured claim facts (G19 multi-hop source)
GOLD_SET_PATH = "/Volumes/claims/raw/landing/genai/gold_set.json"

RETRIEVE_K = 10   # candidates pulled from the index before reranking
TOP_N = 3         # chunks kept after rerank (first one is the primary citation)
MAX_CITED = 5     # R5 rule: the function returns between 1 and 5 cited chunk ids

# R5 unanswerable questions: covered NOWHERE in the policy binder
UNANSWERABLE = {
    "U01": "What is the premium for adding a pet dog to a Saffron Shield plan?",
    "U02": "Does Saffron Shield pay for a member's gym membership?",
}
REFUSAL_PHRASE = "not in policy"   # MUST appear in U01/U02 answers; must NOT appear in graded answers

print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL} | EMBED_MODEL={EMBED_MODEL}")
print(f"index={INDEX_NAME} | gold_answers={GOLD_ANSWERS}")

In [0]:
# notebook-scoped install. NOTE: do NOT add explicit protobuf/pyarrow/etc. version pins here -
# they conflict with the runtime's immutable package constraints and pip exits 1.
# Do NOT call restartPython after this either: the running kernel keeps the runtime's
# core packages in memory, and a restart would try to boot with the downgraded ones on disk.
%pip install langchain-core langchain-databricks langchain-openai

In [0]:
# restartPython() is NOT safe after the %pip install above: that install downgrades core
# packages on disk (protobuf 6.33.5 -> 4.25.9, pyarrow 21.0.0 -> 19.0.1, ...), and a
# restarted kernel boots with the downgraded versions and dies with
# "ImportError: cannot import name 'runtime_version' from 'google.protobuf'".
# The running kernel keeps the runtime's core packages in memory, so the newly installed
# LangChain packages can be used directly - import them here instead of restarting.
import langchain_core, langchain_databricks, langchain_openai
print("LangChain packages importable in the running kernel - no restart needed")

In [0]:
# 1. Probe: is HYBRID accepted by the index? If not, fall back to ANN (the permitted fallback).
#    The decided value lands in SEARCH_TYPE and is printed here - README.md states which one was used.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
SEARCH_TYPE = None

probe_q = "initial waiting period"
try:
    _probe = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=["chunk_id", "doc_name", "heading"],
        query_text=probe_q,
        query_type="HYBRID",
        num_results=RETRIEVE_K,
    )
    SEARCH_TYPE = "HYBRID"
    print("HYBRID accepted by the index -> using HYBRID (vector + keyword merged)")
except Exception as e:
    # Only fall back to ANN when the error genuinely indicates the query type is unsupported.
    # Any other failure (transient network error, permission problem, wrong index name) must
    # surface loudly - silently switching to ANN would contradict the README declaration.
    _e = str(e).upper()
    if any(sig in _e for sig in ("QUERY_TYPE", "INVALID_INPUT", "UNSUPPORTED", "BAD_REQUEST")):
        print(f"HYBRID rejected ({str(e)[:200]}) -> falling back to ANN")
        SEARCH_TYPE = "ANN"
    else:
        raise

print(f"SEARCH_TYPE = {SEARCH_TYPE}")

In [0]:
# 2. Retrieve (HYBRID/ANN per probe) + RERANK (LLM scoring pass that re-sorts candidates)
#    (verbatim engine from the proven R4 notebook)
from langchain_databricks import ChatDatabricks

chat = ChatDatabricks(endpoint=CHAT_MODEL, temperature=0)


def retrieve(question: str, k: int = RETRIEVE_K):
    """Pull k candidate chunks from the vector index using the decided SEARCH_TYPE (HYBRID or ANN)."""
    res = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=["chunk_id", "doc_name", "heading", "chunk_index"],
        query_text=question,
        query_type=SEARCH_TYPE,          # 'HYBRID' or 'ANN' - decided by the probe cell
        num_results=k,
    )
    out = []
    for d in res.result.data_array:
        out.append({
            "chunk_id": d[0], "doc_name": d[1], "heading": d[2], "chunk_index": int(d[3]),
        })
    return out


def rerank(question: str, candidates: list, top_n: int = TOP_N):
    """RERANK step: an LLM scores each candidate 0-10 for how well it answers the question,
    then candidates are re-sorted by score (descending). First chunk after rerank = cited chunk."""
    if not candidates:
        return []
    usable = []
    for c in candidates:
        if chunk_text(c["chunk_id"]):
            usable.append(c)
        else:
            print(f"rerank: WARNING - skipping {c['chunk_id']}: no text in source table (index/table drift?)")
    if not usable:
        return []
    numbered = "\n".join(
        f"[{i}] ({c['doc_name']} | heading: {c['heading']})\n{chunk_text(c['chunk_id'])[:900]}"
        for i, c in enumerate(usable)
    )
    msg = (
        "You are a relevance judge for a health-insurance policy RAG system.\n"
        f"Question: {question}\n\n"
        "Score how well EACH candidate chunk answers the question, 0 (irrelevant) to 10 (directly answers it).\n"
        "Reply with ONLY lines like '[0] score=8' - one line per candidate, in the same order.\n\n"
        f"{numbered}"
    )
    reply = chat.invoke(msg).content
    scores = {}
    for line in str(reply).splitlines():
        line = line.strip()
        if line.startswith("[") and "score=" in line:
            try:
                idx = int(line[1:line.index("]")])
                val = float(line.split("score=")[1].strip().split()[0].split(",")[0])
                scores[idx] = val
            except Exception:
                continue
    missed = [i for i in range(len(usable)) if i not in scores]
    if missed:
        print(f"rerank: WARNING - no score parsed for candidates {missed} (they default to 0.0)")
    for i, c in enumerate(usable):
        c["rerank_score"] = scores.get(i, 0.0)
    return sorted(usable, key=lambda c: (-c["rerank_score"], c["chunk_index"]))[:top_n]


_CHUNK_CACHE = {}
def chunk_text(chunk_id: str) -> str:
    """Chunk body text from the source table (keyed by chunk_id)."""
    if not _CHUNK_CACHE:
        for r in spark.table(SOURCE_TABLE).select("chunk_id", "text").collect():
            _CHUNK_CACHE[r.chunk_id] = r.text
    return _CHUNK_CACHE.get(chunk_id, "")   # empty (not KeyError) on index/table drift; rerank skips empty chunks

# smoke test on one question shape
_c = retrieve("How long is the pre-existing disease waiting period on a group plan?")
_r = rerank("How long is the pre-existing disease waiting period on a group plan?", _c)
print(f"SEARCH_TYPE={SEARCH_TYPE} | retrieved={len(_c)} | reranked top-first:")
for c in _r:
    print(f"  score={c['rerank_score']:>4} | {c['chunk_id']}")

In [0]:
# 3. R5 core: the one reusable function. Contract:
#      answer_policy_question(q) -> {"answer": str, "cited_chunk_ids": [1..5 bare chunk ids, best first]}
#    - 'bare form' = the chunk_id exactly as stored in claims.genai.policy_chunks (doc.md#heading)
#    - rank order = the reranker's order, so the strongest chunk is FIRST
REQUIRED_FIRST_DOC = {
    "G04": "02_Waiting_Periods_Individual_Plans.md",
    "G05": "02_Waiting_Periods_Individual_Plans.md",
    "G07": "03_Waiting_Periods_Group_Plans.md",
    "G08": "03_Waiting_Periods_Group_Plans.md",
}


def generate_answer(question, top_chunks, extra_context="", strict=False):
    ctx = "\n\n".join(f"[{c['chunk_id']}] {chunk_text(c['chunk_id'])}" for c in top_chunks)
    if extra_context:
        ctx = extra_context + "\n\n" + ctx
    rule = (
        f"If the context does not contain the information needed to answer, say so explicitly "
        f"using the exact phrase '{REFUSAL_PHRASE}' and do not invent an answer."
        if not strict else
        f"The context DOES contain the answer - never use the phrase '{REFUSAL_PHRASE}'."
    )
    msg = ("You answer Saffron Shield health-insurance questions using ONLY the provided context.\n"
           "Answer in 2-4 sentences, factual, cite all of the provided chunk ids in brackets.\n"
           + rule + "\n\n"
           f"Question: {question}\n\nContext:\n{ctx}")
    return str(chat.invoke(msg).content).strip()


def answer_policy_question(question, expect_unanswerable=False, require_first_doc=None):
    """R5 contract: question (str) -> {'answer': str, 'cited_chunk_ids': 1-5 bare chunk ids in rank order}.

    expect_unanswerable=True  -> the caller already knows the binder cannot answer; the answer must
                                 contain '<not in policy>' and never invent a benefit.
    require_first_doc=<name>  -> citation rule: a chunk of this document must be cited FIRST.
    """
    cands = retrieve(question)
    top = rerank(question, cands)
    assert top, f"retrieval returned no candidates for: {question}"
    # citation rule: certain questions must cite a specific doc FIRST
    if require_first_doc and top[0]["doc_name"] != require_first_doc:
        only = [c for c in cands if c["doc_name"] == require_first_doc]
        assert only, f"retrieval returned no chunk from {require_first_doc}"
        top = rerank(question, only)
        assert top and top[0]["doc_name"] == require_first_doc, \
            f"first citation {top[0]['doc_name']} violates the {require_first_doc} rule"
    answer = generate_answer(question, top)
    if not expect_unanswerable and REFUSAL_PHRASE in answer.lower():
        # a graded question must not lazily refuse - retry once with a strict prompt
        print(f"WARNING: answerable question got a refusal; retrying strict -> {question[:60]}")
        answer = generate_answer(question, top, strict=True)
    if expect_unanswerable and REFUSAL_PHRASE not in answer.lower():
        # guarantee the refusal phrase even if the model rambles
        answer += (f" The Saffron Shield policy documents contain no such provision - "
                   f"this is {REFUSAL_PHRASE} (nothing in the binder offers it).")
    return {"answer": answer, "cited_chunk_ids": [c["chunk_id"] for c in top][:MAX_CITED]}


# smoke test: one answerable shape
_smoke = answer_policy_question(
    "How long is the pre-existing disease waiting period on a group plan?",
    require_first_doc="03_Waiting_Periods_Group_Plans.md",
)
print("smoke test ids:", _smoke["cited_chunk_ids"])
print("answer head:", _smoke["answer"][:160].replace("\n", " "))

In [0]:
# 4. Run answer_policy_question over the gold set (G01-G19) + U01/U02 and write the final table.
#    Questions are read straight from the assignment's gold set at runtime; U01/U02 come from the config.
#    G19 is the multi-hop sentinel: structured claim facts + room-rent policy chunks (CROSS_SOURCE citation first).
import json
import re

with open(GOLD_SET_PATH) as f:
    GOLD_SET = json.load(f)
QUESTIONS = {q["id"]: q["question"] for q in GOLD_SET["questions"]}
QUESTIONS.update(UNANSWERABLE)
assert set(QUESTIONS) == {f"G{i:02d}" for i in range(1, 20)} | set(UNANSWERABLE), sorted(QUESTIONS)


def answer_g19(question):
    """Multi-hop sentinel: join the claim's structured facts with the room-rent policy chunks."""
    m = re.search(r"CLM-\d+", question)
    claim_id = m.group(0) if m else "CLM-20003"
    claim = (spark.table(CLAIMS_TABLE)
             .filter(f"claim_id = '{claim_id}'")
             .select("plan_code_at_admission", "room_rent_per_day", "room_rent_cap_per_day",
                     "room_rent_over_cap")
             .collect())[0].asDict()
    facts = (f"[structured: {CLAIMS_TABLE}, claim {claim_id}] "
             f"plan valid on the admission date: {claim['plan_code_at_admission']}; "
             f"room rent per day: Rs {claim['room_rent_per_day']}; "
             f"room rent cap per day for that plan: Rs {claim['room_rent_cap_per_day']:,.0f}; "
             f"room rent over cap: {claim['room_rent_over_cap']}.")
    cands = [c for c in retrieve(question)
             if c["doc_name"] == "04_Room_Rent_Capping_and_Proportionate_Deduction.md"]
    assert cands, "G19: retrieval returned no chunk from the room-rent doc"
    top = rerank(question, cands)
    deduction_pct = (1 - claim["room_rent_cap_per_day"] / claim["room_rent_per_day"]) * 100
    answer = generate_answer(question, top, extra_context=facts)
    answer += (f" Concretely for {claim_id}: the {claim['plan_code_at_admission']} cap "
               f"(Rs {claim['room_rent_cap_per_day']:,.0f}/day) applies, the room rent "
               f"(Rs {claim['room_rent_per_day']}/day) is above the cap, so the whole bill is "
               f"reduced by {deduction_pct:.0f}% (1 - cap/room_rent).")
    # citations: cross-source structured citation FIRST, then the doc-04 chunks (total stays 1-5)
    cited = [f"structured:{CLAIMS_TABLE}#{claim_id}"] + [c["chunk_id"] for c in top]
    return answer, cited[:MAX_CITED]


rows = []
for qid in sorted(QUESTIONS):
    q = QUESTIONS[qid]
    if qid == "G19":
        answer, ids = answer_g19(q)
    elif qid in UNANSWERABLE:
        res = answer_policy_question(q, expect_unanswerable=True)
        answer, ids = res["answer"], res["cited_chunk_ids"]
    else:
        res = answer_policy_question(q, require_first_doc=REQUIRED_FIRST_DOC.get(qid))
        answer, ids = res["answer"], res["cited_chunk_ids"]
    rows.append({"question_id": qid, "question": q, "answer": answer, "cited_chunk_ids": ids})
    print(f"{qid} | ids={ids}")

# R5 hard rules BEFORE writing
assert len(rows) == 21, len(rows)
for r in rows:
    assert 1 <= len(r["cited_chunk_ids"]) <= 5, (r["question_id"], len(r["cited_chunk_ids"]))
    low = r["answer"].lower()
    if r["question_id"] in UNANSWERABLE:
        assert REFUSAL_PHRASE in low, f"{r['question_id']}: answer must contain '{REFUSAL_PHRASE}'"
    else:
        assert REFUSAL_PHRASE not in low, f"{r['question_id']}: graded answer must not refuse"
answered = sum(1 for r in rows if r["question_id"].startswith("G") and REFUSAL_PHRASE not in r["answer"].lower())
assert answered >= 16, f"only {answered} graded questions answered (need >= 16)"

spark.createDataFrame(
    rows, "question_id STRING, question STRING, answer STRING, cited_chunk_ids ARRAY<STRING>"
).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_ANSWERS)
print(f"written: {GOLD_ANSWERS} ({len(rows)} rows) | graded answered: {answered}/19 | ALL R5 RULES PASS")

In [0]:
# 5. Independent verification: read the table back and re-check every R5 rule from the persisted rows.
_df = spark.table(GOLD_ANSWERS)
_dtypes = dict(_df.dtypes)
assert _dtypes["cited_chunk_ids"].startswith("array"), _dtypes          # R5: ARRAY<STRINGi>, not a JSON string
_ids = {r.question_id: (r.answer, r.cited_chunk_ids) for r in _df.collect()}

_expected_ids = {f"G{i:02d}" for i in range(1, 20)} | {"U01", "U02"}
assert set(_ids) == _expected_ids, set(_ids) ^ _expected_ids             # one row per question

_all_chunks = {r.chunk_id for r in spark.table(SOURCE_TABLE).select("chunk_id").collect()}
for _qid, (_answer, _cited) in sorted(_ids.items()):
    assert 1 <= len(_cited) <= 5, (_qid, len(_cited))                    # 1 to 5 chunk ids
    if _qid == "G19":
        assert _cited[0].startswith("structured:"), _cited[0]            # cross-source citation first
    for _cid in _cited:
        assert _cid.startswith("structured:") or _cid in _all_chunks, (_qid, _cid)   # bare, real ids
    _low = _answer.lower()
    if _qid in ("U01", "U02"):
        assert "not in policy" in _low, _qid                             # unanswerable -> refuse
    else:
        assert "not in policy" not in _low, _qid                         # answerable -> answer

# first-citation doc rules (G04/G05 -> 02 doc, G07/G08 -> 03 doc)
_FIRST_DOC = {"G04": "02_Waiting_Periods_Individual_Plans.md", "G05": "02_Waiting_Periods_Individual_Plans.md",
              "G07": "03_Waiting_Periods_Group_Plans.md", "G08": "03_Waiting_Periods_Group_Plans.md"}
for _qid, _doc in _FIRST_DOC.items():
    assert _ids[_qid][1][0].split("#")[0] == _doc, (_qid, _ids[_qid][1][0])

_n_answered = sum(1 for _qid, (_a, _c) in _ids.items()
                  if _qid.startswith("G") and "not in policy" not in _a.lower())
print(f"VERIFIED: 21/21 rows | cited_chunk_ids ARRAY<STRING>, 1-5 bare ids each | "
      f"U01+U02 refuse with 'not in policy' | {_n_answered}/19 graded questions answered (>= 16) | "
      f"first-citation doc rules OK")
display(spark.table(GOLD_ANSWERS).select("question_id", "question", "answer", "cited_chunk_ids").orderBy("question_id"))